# Imports and Setup

In [ ]:
%load_ext autoreload

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s') # NOTSET, DEBUG, INFO, WARN, ERROR, CRITICAL

import os
import numpy as np

os.environ["JAX_PLATFORM_NAME"] = "cpu"  # must be set before importing jax

import jax
import jax.numpy as jnp
from jax import vmap, jit, grad, jacfwd

# Optional: higher numeric accuracy (do this before heavy JAX use)
# try:
#     from jax import config as jax_config
#     jax_config.update("jax_enable_x64", True)
# except Exception:
#     pass

import skewerjax as sjx

import matplotlib as mpl
from matplotlib.collections import LineCollection
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import matplotlib.ticker as mticker
plt.style.use('tableau-colorblind10')
font, rcnew = sjx.config.setup_matplotlib.matplotlib_default_config()
plt.rc('font', **font)
plt.rcParams.update(rcnew)
# %matplotlib widget
%matplotlib inline

In [ ]:
from skewerjax.skewers.skewers import SkewerBundle
from skewerjax.cosmology.cosmology import Cosmology

from skewerjax.fields.fields import ScalarField, VectorField, Fields
import skewerjax.fields.toy_model_samplers as toy_samplers
import skewerjax.utils.aux as aux
import skewerjax.utils.plotting_tools as pt

import skewerjax.optical_depth.optical_depth as od

from skewerjax.constants.physical_constants import phys

# Cosmology

In [ ]:
# Create cosmology object
cosmology = Cosmology(H0=phys.H0, Omega_r_0=0.0, Omega_m_0=1.0, Omega_k_0=0.0, Omega_lambda_0=0.0)
# Define which cosmology to use, this is set as default to 0 since skewer and cosmology objects can deal with multiple cosmologies at once, 
# but other classes such as fields and optical depth are currently only set up for a single cosmology at a time
ii_cosmology = 0

# Lines

In [ ]:
line_bundle = sjx.constants.lines.LineBundle(["HI-LyA", "SiIII-1206"], ref_line_id = "HI-LyA") # "SiIII-1206", "SiII-1190", "SiII-1193" # "HI-LyA-1206", "HI-LyA-1193", "HI-LyA-1190"

# Skewers

In [ ]:
L_box = 110.7  # Mpc
z_box = 2.0

In [ ]:
N_grid_skewers = 5
N_ppsk = 4096

points_grid_skewers = aux.make_skewer_grid_points(L_box, N_grid_skewers, N_ppsk, direction="y")
points = jnp.reshape(points_grid_skewers, (N_grid_skewers**2, N_ppsk, 3))

In [ ]:
skewer_bundle = SkewerBundle(spatial_points = points, cosmologies = cosmology, a0 = 1/(1+z_box), x0 = L_box)

# Fields

In [ ]:
N_absorvers = 3
Delta = 0.5

n0_HI = 1e-11
T0_HI = 3e3

n0_SiIII = 1e-11
T0_SiIII = 3e3

n0_SiII = 1e-11
T0_SiII = 3e3

fields = Fields({
    "HI": {
        "temperature": ScalarField(toy_samplers.periodic_top_hat_absorber_sampler(N_absorvers=N_absorvers, Delta=Delta, C_low=1e-5, C0=T0_HI, L_box=L_box, base_seed=0)),    # UNITS: [K]
        "number_density": ScalarField(toy_samplers.periodic_top_hat_absorber_sampler(N_absorvers=N_absorvers, Delta=Delta, C_low=1e-20, C0=n0_HI, L_box=L_box, base_seed=0)), # [cm^-3]
        "velocity": VectorField(toy_samplers.velocity_sampler(0.0, 0.0, L_box)),        # UNITS: [km/s]
        "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),         # UNITS: [km/s]
        "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0))        # UNITS: [GHz]
    },
    "SiIII": {
        "temperature": ScalarField(toy_samplers.periodic_top_hat_absorber_sampler(N_absorvers=N_absorvers, Delta=Delta, C_low=1e-5, C0=T0_SiIII, L_box=L_box, base_seed=0)),    # UNITS: [K]
        "number_density": ScalarField(toy_samplers.periodic_top_hat_absorber_sampler(N_absorvers=N_absorvers, Delta=Delta, C_low=1e-20, C0=n0_SiIII, L_box=L_box, base_seed=0)), # [cm^-3]
        "velocity": VectorField(toy_samplers.velocity_sampler(0.0, 0.0, L_box)),        # UNITS: [km/s]
        "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),         # UNITS: [km/s]
        "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0))        # UNITS: [GHz]
    },
    # "SiII": {
    #     "temperature": ScalarField(toy_samplers.periodic_top_hat_absorber_sampler(N_absorvers=N_absorvers, Delta=Delta, C_low=1e-5, C0=T0_SiII, L_box=L_box, base_seed=0)),    # UNITS: [K]
    #     "number_density": ScalarField(toy_samplers.periodic_top_hat_absorber_sampler(N_absorvers=N_absorvers, Delta=Delta, C_low=1e-20, C0=n0_SiII, L_box=L_box, base_seed=0)), # [cm^-3]
    #     "velocity": VectorField(toy_samplers.velocity_sampler(0.0, 0.0, L_box)),        # UNITS: [km/s]
    #     "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),         # UNITS: [km/s]
    #     "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0))        # UNITS: [GHz]
    # },
})

In [ ]:
# T0_HI = 1e5
# T0_Si = 2e6
# n0_min = 1e-16
# n0_max = 1e-10
# def pk_template(k: jnp.ndarray, A, n, k0, k_cut):
#     # P(k) = A (k/k0)^n exp[-(k/k_cut)^2]
#     kk = jnp.maximum(k, 1e-12)
#     return A * (kk / k0) ** n * jnp.exp(-(kk / k_cut) ** 2)
# number_density_sampler = toy_samplers.number_density_sampler_gaussian_3d(
#     key=jax.random.PRNGKey(0),
#     pk_of_k=lambda k: pk_template(k, A=5.0, n=-2.5, k0=2.0, k_cut=50.0),
#     n0_min=n0_min, n0_max=n0_max, L_box=L_box,
#     grid_shape=(256, 256, 256), contrast=1.2,
# )
# fields = Fields({
#     "HI": {
#         "temperature": ScalarField(toy_samplers.temperature_sampler(T0_HI, T0_HI, L_box)),    # UNITS: [K]
#         "velocity": VectorField(toy_samplers.velocity_sampler(0.0, 0.0, L_box)),        # UNITS: [km/s]
#         "number_density": ScalarField(number_density_sampler),                          # [cm^-3]
#         "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),         # UNITS: [km/s]
#         "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0))        # UNITS: [GHz]
#     },
#     "SiIII": {
#         "temperature": ScalarField(toy_samplers.temperature_sampler(T0_Si, T0_Si, L_box)),    # UNITS: [K]
#         "velocity": VectorField(toy_samplers.velocity_sampler(0.0, 0.0, L_box)),        # UNITS: [km/s]
#         "number_density": ScalarField(number_density_sampler),                          # [cm^-3]
#         "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),         # UNITS: [km/s]
#         "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0))        # UNITS: [GHz]
#     },
#     # "SiII": {
#     #     "temperature": ScalarField(toy_samplers.temperature_sampler(T0_Si, T0_Si, L_box)),    # UNITS: [K]
#     #     "velocity": VectorField(toy_samplers.velocity_sampler(0.0, 0.0, L_box)),        # UNITS: [km/s]
#     #     "number_density": ScalarField(number_density_sampler),                          # [cm^-3]
#     #     "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),         # UNITS: [km/s]
#     #     "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0))        # UNITS: [GHz]
#     # },
# })

### Plot Fields

In [ ]:
sampled_fields = fields.sample(skewer_bundle.quadri_vectors[ii_cosmology], skewer_bundle.los_directions)

In [ ]:
x_array = points[..., 1]

key_field_plot = 'number_density'
plot_dict = {}
for ii, key in enumerate(sampled_fields.keys()):
    plot_dict[key] = jnp.log10(sampled_fields[key][key_field_plot])

fig, ax = pt.plot_3d_radar_cylinder(
    x_array=x_array, y_dict=plot_dict, cylinder_radius=12.0, skewer_index=0, field_logscale=False,
    x_axis_label=r"$r_{\parallel}\,[{\rm cMpc}]$",
    field_units=r"$\log_{10}\left(n \, \left[\mathrm{cm}^{-3}\right] \right)$",
)

fig, ax = pt.plot_3d_neon_ridge_cube(
    points=points, y_dict=plot_dict, normalization="global", field_logscale=False, normalize_scales=False,
    x_label=r"$x\,[{\rm cMpc}]$", y_label=r"$r_{\parallel}\,[{\rm cMpc}]$", z_label=r"$z\,[{\rm cMpc}]$",
    axis_label_fontsize=22, x_label_pad = 25.0, y_label_pad = 25.0, z_label_pad = 25.0, n_axis_ticks=(4, 4, 4),
    ridge_scale=1.0,
)

# Optical depth

In [ ]:
a_dagger = 1/(1+z_box)

v_max = phys.c_kms * jnp.log(a_dagger / skewer_bundle.scale_factors[0, 0, 0])
velocities_kms = jnp.linspace(0.0, v_max, 1024)

optical_depth = od.compute_optical_depth(
    skewer_bundle=skewer_bundle, velocities_kms=velocities_kms, line_bundle=line_bundle, fields=fields, a_dagger=a_dagger,
)
optical_depth_wrapped = od.wrap_optical_depth_to_ref_line(line_bundle=line_bundle, velocities_kms=velocities_kms, optical_depth=optical_depth)

### Plot optical depth

In [ ]:
x_array = jnp.tile(velocities_kms[None], (N_grid_skewers**2, 1))

plot_dict = {}
for ii, line in enumerate(line_bundle.lines):
    plot_dict[line.name] = jnp.log10(optical_depth_wrapped[ii])
    
fig, ax = pt.plot_3d_radar_cylinder(
    x_array=x_array, y_dict=plot_dict, cylinder_radius=2000.0, skewer_index=0, field_logscale=False,
    factor_ticks_x=3.0, factor_ticks_z=1200.0, factor_x_label=0.8, x_axis_label=r"$v\,[{\rm km / s}]$",
    field_units=r"$\log_{10}\tau$",
)

N_skewers = optical_depth.shape[1]
N_velocity = velocities_kms.size

x_trans = points[:N_skewers, 0, 0]
z_trans = points[:N_skewers, 0, 2]

x_cube = jnp.tile(x_trans[:, None], (1, N_velocity))
y_cube = jnp.tile(velocities_kms[None, :], (N_skewers, 1))
z_cube = jnp.tile(z_trans[:, None], (1, N_velocity))

points_cube_velocities = jnp.stack([x_cube, y_cube, z_cube], axis=-1)

fig, ax = pt.plot_3d_neon_ridge_cube(
    points=points_cube_velocities, y_dict=plot_dict, normalization="global", field_logscale=False, normalize_scales=True,
    x_label=r"$x\,[{\rm cMpc}]$", y_label=r"$v\,[{\rm km\,s^{-1}}]$", z_label=r"$z\,[{\rm cMpc}]$", axis_label_fontsize=22,
    ridge_scale=1.0, x_label_pad = 25.0, y_label_pad = 25.0, z_label_pad = 25.0, n_axis_ticks=(4, 4, 4),
)

fig, axes, idx = pt.plot_transmission_samples_neon(
    F_over_F0=np.exp(-optical_depth_wrapped), line_bundle=line_bundle, x_axis_array=velocities_kms, x_label=r"$v \, \left[\mathrm{km} \, \mathrm{s}^{-1}\right]$", x_axis_key="velocities_kms",
    scale_factor_left=skewer_bundle.scale_factors[0, 0, 0], scale_factor_right=skewer_bundle.scale_factors[0, 0, -1], lambda0_anchor=None, N_plot_sample=4, seed=0
)

# Overflux

In [ ]:
overflux = sjx.optical_depth.overflux.DeltaFieldBuilder(optical_depth_wrapped).build()

In [ ]:
Npix = optical_depth_wrapped.shape[-1]

delta_v_list = list(np.array(line_bundle.delta_v))
L = np.max(velocities_kms) - np.min(velocities_kms)

dv = np.diff(velocities_kms)[0]
stats = sjx.summary_stats.P1D_xi1D.OverfluxStatistics1D(overflux, dv)
ordered_pairs = stats.ordered_pairs

In [ ]:
stats.compute_p1d()

Pk_total_from_terms = stats.Pk_total_from_terms

Pk_total_per_sample = stats.Pk_total_per_sample
Pk_total = stats.Pk_total

Pk_pair_per_sample = stats.Pk_pair_per_sample
Pk_pair = stats.Pk_pair

k = stats.k

In [ ]:
# ================================================================
# Plot P1D

fig, axes, axes_res, order_sum_dict = sjx.utils.plotting_tools_P1D_xi1D.plot_p1d_by_order(
    ordered_pairs=ordered_pairs, Pk_pair=Pk_pair, k=k, delta_v_list=delta_v_list, L_box=L, Npix=Npix,
    tolerance_threshold=1e-6,
    use_log_x=True,
    # xlim=(2.0, 600),
    residual_mode="sum",
    show_harmonics=True,
    use_symlog_main=True,
    use_symlog_residual=False,
    symlog_linscale=1.0,
    symlog_label_minor=False,
    symlog_sci_cutoff=1e-2,
    legend_ncol_terms=2,
    xlabel=r"$k\, L_\mathrm{box} / (2\pi)$", ylabel=r"$P_{\rm 1D}$", residual_ylabel=r"$\Delta$",
)

fig, (ax, ax_res), order_sum_dict = sjx.utils.plotting_tools_P1D_xi1D.plot_p1d_total_recovery(
    Pk_total=Pk_total, Pk_total_from_terms=Pk_total_from_terms, Pk_pair=Pk_pair, k=k, delta_v_list=delta_v_list, L_box=L, Npix=Npix,
    ordered_pairs=ordered_pairs,
    show_harmonics=False,
    show_order_sums=True,
    show_order_sum_residuals=True,
    use_symlog_main=True,
    use_symlog_residual=False,
    xlabel=r"$k\, L_\mathrm{box} / (2\pi)$"
)

In [ ]:
# ===============================================================
# Build order sums from Pk_pair

ordered_pairs = list(Pk_pair.keys()) if ordered_pairs is None else ordered_pairs
order_sum_dict = {}
for pair in ordered_pairs:
    A, B = pair
    order = len(A) + len(B)
    if order not in order_sum_dict:
        order_sum_dict[order] = np.zeros_like(Pk_pair[pair], dtype=float)
    order_sum_dict[order] += Pk_pair[pair].real
orders = sorted(order_sum_dict.keys())

# ===============================================================
# Plot P1D

cmap = plt.get_cmap("tab10")

fig, ax = plt.subplots(figsize=(12, 9))
for i, order in enumerate(orders):
    y_order = order_sum_dict[order][1:] * k[1:] / np.pi
    ax.plot(k[1:], y_order, lw=2.0, color=cmap(i % 10), alpha=0.9, label=rf"$\mathcal{{O}}({order})$")

# Total
ax.plot(k[1:], Pk_total.real[1:] * k[1:] / np.pi, lw=3.0, color="k", label=r"Tot.")

# Total recovered from terms
ax.plot(k[1:], Pk_total_from_terms.real[1:] * k[1:] / np.pi, lw=2.0, color="red", ls="--", label=r"Tot. Sum")

ax.set_yscale("log")

ax.set_xlabel(r"$k$", fontsize=22)
ax.set_ylabel(r"$k P_{\rm 1D}(k) / \pi$", fontsize=22)

ax.set_xlim([0.0, 0.045])
ax.set_ylim([0.007, 0.6])

ax.tick_params(axis="both", labelsize=20)
ax.legend(fontsize=20, ncol=2)

plt.tight_layout()
plt.show()

In [ ]:
stats.compute_xi1d()

xi_total_from_terms = stats.xi_total_from_terms

xi_total_per_sample = stats.xi_total_per_sample
xi_total = stats.xi_total

xi_pair_per_sample = stats.xi_pair_per_sample
xi_pair = stats.xi_pair

lags = stats.lags
lags_norm = lags / L

In [ ]:
# ================================================================
# Plot xi1D

fig, axes, axes_res, xi_order_sum_dict, xi_order_sum_plot_dict = sjx.utils.plotting_tools_P1D_xi1D.plot_xi_by_order(
    ordered_pairs=ordered_pairs, xi_pair=xi_pair, lags_norm=lags_norm, delta_v_list=delta_v_list, L_box=L,
    tolerance_threshold=1e-6,
    residual_mode="sum",
    xlim=(-0.0, 0.5),
    show_delta_v_diff_lines=True,
    use_symlog_main=False,
    use_symlog_residual=False,
    symlog_linscale=1.0,
    symlog_base=10,
    symlog_label_minor=False,
    symlog_sci_cutoff=1e-2,
    show_order_badge=True,
    legend_ncol_terms=2,
    xlabel=r"$\Delta v / L_\mathrm{box}$", ylabel=r"$\xi_{\rm 1D}$", residual_ylabel=r"$\Delta$",
)

fig, (ax, ax_res), xi_order_sum_dict, xi_order_sum_plot_dict = sjx.utils.plotting_tools_P1D_xi1D.plot_xi_total_recovery(
    xi_total=xi_total, xi_total_from_terms=xi_total_from_terms, xi_pair=xi_pair, lags_norm=lags_norm, delta_v_list=delta_v_list, L_box=L, ordered_pairs=ordered_pairs,
    xlim=(-0.0, 0.5),
    show_order_sums=True,
    show_order_sum_residuals=True,
    show_delta_v_diff_lines=True,
    annotate_half_line=False,
    half_line_text=r"$L_\mathrm{box}/2$",
    use_symlog_main=False,
    use_symlog_residual=True,
    main_symlog_linthresh=None,
    symlog_linscale=1.0,
    symlog_base=10,
    symlog_sci_cutoff=1e-2,
    xlabel=r"$\Delta v / L_\mathrm{box}$", ylabel=r"$\xi_{\rm 1D}$", residual_ylabel=r"$\Delta$",
)